# Python Program to Check Plagiarism\n\nThis notebook compares `.txt` files in the `documents/` folder using TF-IDF and cosine similarity.\n\n**Note:** Similarity scores help identify documents for review; they do not prove plagiarism.

In [ ]:
from pathlib import Path
from itertools import combinations

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# The notebook reads .txt files from the project's documents/ folder.
DOCUMENTS_DIR = Path("documents")

def load_documents(folder=DOCUMENTS_DIR):
    """Load all UTF-8 .txt documents from a folder."""
    return {
        path.name: path.read_text(encoding="utf-8")
        for path in sorted(folder.glob("*.txt"))
    }

def check_plagiarism(documents):
    """Return document pairs and their cosine similarity scores."""
    if len(documents) < 2:
        return []

    names = list(documents)
    texts = [documents[name] for name in names]
    tfidf_matrix = TfidfVectorizer().fit_transform(texts)
    similarity_matrix = cosine_similarity(tfidf_matrix)

    results = []
    for i, j in combinations(range(len(names)), 2):
        results.append((names[i], names[j], similarity_matrix[i, j]))
    return sorted(results, key=lambda row: row[2], reverse=True)

documents = load_documents()

if len(documents) < 2:
    print(f"Found {len(documents)} .txt file(s) in {DOCUMENTS_DIR.resolve()}")
    print("Add at least two text files to compare.")
else:
    print(f"Comparing {len(documents)} documents")
    print("-" * 65)
    for file_a, file_b, score in check_plagiarism(documents):
        print(f"{file_a}  <->  {file_b}: {score:.1%}")
    print("\nSimilarity scores are indicators for human review, not proof of plagiarism.")
